# DocTR OCR Testing

**objective:**
testing docTR's capabilities to extract text from images and pdfs in one shot, meaning without having to do any cropping nor background removal, on two types of docs: CINs, and Papyrus.

## Setup

In [ ]:
import io
import time
from pathlib import Path

import fitz
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
from doctr.io import DocumentFile
from doctr.models import ocr_predictor
from PIL import Image

## Helpers

In [ ]:
# Define CIN image files
data_dir = Path("../data")
cin_files = [
    data_dir / "trash_cin.jpg",
    data_dir / "trash_cin_1.jpg",
    data_dir / "trash_cin_1.pdf",
    data_dir / "trash_cin_2.jpg",
    data_dir / "trash_cin_2.pdf",
]

print(f"Found {len(cin_files)} CIN files to process")
for f in cin_files:
    print(f"  - {f.name} ({f.stat().st_size / 1024:.1f} KB)")

In [ ]:
def plot_doc(path):
    """Plot image or PDF file based on file suffix"""
    file_extension = path.suffix.lower()
    if file_extension in [".png", ".jpg", ".jpeg", ".bmp", ".tiff", ".gif", ".webp"]:
        # Display image
        try:
            img = mpimg.imread(str(path))
            plt.figure(figsize=(10, 8))
            plt.imshow(img)
            plt.title(f"Image: {path.name}")
            plt.axis("off")
            plt.tight_layout()
            plt.show()
            return True
        except Exception as e:
            print(f"Could not display image: {e}")
            return False

    elif file_extension == ".pdf":
        # Display PDF (first page)
        try:
            pdf_document = fitz.open(str(path))
            if pdf_document.page_count > 0:
                page = pdf_document.load_page(0)
                pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))  # 2x zoom for better quality

                # Convert to image
                img_data = pix.tobytes("png")
                img = Image.open(io.BytesIO(img_data))

                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.title(f"PDF (Page 1 of {pdf_document.page_count}): {path.name}")
                plt.axis("off")
                plt.tight_layout()
                plt.show()

                print(f"PDF has {pdf_document.page_count} page(s)")
                pdf_document.close()
                return True
            else:
                print("PDF has no pages")
                pdf_document.close()
                return False
        except Exception as e:
            print(f"Could not display PDF: {e}")
            return False

    else:
        print(f"Unsupported file type for preview: {file_extension}")
        return False

In [ ]:
def load_document(path):
    """Load document using docTR's DocumentFile based on file type"""
    file_extension = path.suffix.lower()
    if file_extension == ".pdf":
        return DocumentFile.from_pdf(str(path))
    else:
        return DocumentFile.from_images(str(path))

In [ ]:
def convert(predictor, path, show_doc=True):
    """show image, compute conversion or OCR time, outputs result"""
    print(f"================== Processing {path.name}... ==================")
    if show_doc:
        plot_doc(path)

    # Load document
    doc = load_document(path)

    tic = time.time()
    result = predictor(doc)
    toc = time.time()
    print(f"================== Processing time: ({toc-tic:.2f}s) ==================")

    # Extract text from docTR result
    text_items = []
    for page in result.pages:
        for block in page.blocks:
            for line in block.lines:
                line_text = " ".join([word.value for word in line.words])
                text_items.append(line_text)

    text_str = "\n".join(text_items)
    print(f"================== Results: ==================\n {text_str}")
    return result

## DocTR's Default OCR on CINs

In [ ]:
# Initialize docTR predictor with default models
predictor = ocr_predictor(pretrained=True)

In [ ]:
convert(predictor, cin_files[0])

In [ ]:
convert(predictor, cin_files[1])

In [ ]:
convert(predictor, cin_files[2])

In [ ]:
convert(predictor, cin_files[3])

In [ ]:
convert(predictor, cin_files[4])

**Takeways:**

- doesn't seem so much different from docling, neither on time nor quality, both get some correct and get some wrong

## DocTR's default OCR on entire papyrus

In [ ]:
convert(predictor, data_dir / "trash_papyrus_1.pdf", show_doc=False)

In [ ]:
convert(predictor, data_dir / "trash_papyrus_2.pdf", show_doc=False)

## Improving on DocTR's Default OCR

In [ ]:
# Initialize docTR predictor with different model architectures
# Detection: db_resnet50, db_mobilenet_v3_large
# Recognition: crnn_vgg16_bn, crnn_mobilenet_v3_small, master, sar_resnet31
custom_predictor = ocr_predictor(det_arch="db_resnet50", reco_arch="crnn_vgg16_bn", pretrained=True)

In [ ]:
convert(custom_predictor, cin_files[1])

In [ ]:
convert(custom_predictor, cin_files[2])

**Takeaways:**

- doctr's option still need to be explored further, it has plenty of options

- the execution time on papyrus is quite long compared to docling